# Goldman SDD Algorithm — Python Port
### Google Earth Engine Python API

Direct Python translation of the Goldman Snow Disappearance Day (SDD) script.
Produces one 500-m GeoTIFF per year (2024, 2025) exported to Google Drive.

Variable names, logic, and order match the original JavaScript exactly.

## Cell 1 — Authenticate

In [2]:
# !pip install earthengine-api --quiet   # uncomment if needed

import ee
import geopandas as gpd

ee.Authenticate()
ee.Initialize(project='ee-jandrewgoldman')   # ← replace

print('Earth Engine initialised.')

Earth Engine initialised.


## Cell 2 — User Inputs

Edit this cell. All variable names match the Goldman script.

In [3]:
# ── Region asset ──────────────────────────────────────────────────────────────
# Your study region as a GEE FeatureCollection asset.
# Goldman uses individual fire perimeters — we use the full region boundary
# and export one raster covering the entire area.
#Read local file and reproject to WGS84
gdf = gpd.read_file("../data/processed_study_zones/study_zones_wgs.geojson").to_crs("EPSG:4326")
region_fc = ee.FeatureCollection(gdf.__geo_interface__)
region_geom = region_fc.geometry()

# ── Years to run ──────────────────────────────────────────────────────────────
YEARS = [2024,2025]

# ── Goldman user inputs (named identically to original) ──────────────────────
# Goldman: var wyStartDay = '-10-01'
# We use a spring melt window instead of a full water year to reduce compute.
# Adjust these dates for your region's typical melt season.
wyStartDay    = '-02-01'   # start of primary search window (Goldman: '-10-01')
wyEndDay      = '-08-01'   # end of primary search window

# Goldman: var NDSI_threshold = '15'
NDSI_threshold = 15

# Goldman: var low_value = 0.07
low_value = 0.07

# Goldman: var prior_days = 5
prior_days = 5

# Goldman: var minNoSnowdays = 5
minNoSnowdays = 5

# Goldman: var projection = 'EPSG:4326'
projection = 'EPSG:4326'

# ── Export folder ─────────────────────────────────────────────────────────────
imagesFolder = 'SDDperYear'   # Goldman: var imagesFolder = 'SDDperFire'

print('User inputs set.')
print(f'  NDSI_threshold : {NDSI_threshold}')
print(f'  low_value      : {low_value}')
print(f'  prior_days     : {prior_days}')
print(f'  minNoSnowdays  : {minNoSnowdays}')
print(f'  Window         : {wyStartDay} → {wyEndDay} + 30 adv. days')

User inputs set.
  NDSI_threshold : 15
  low_value      : 0.07
  prior_days     : 5
  minNoSnowdays  : 5
  Window         : -02-01 → -08-01 + 30 adv. days


## Cell 3 — Load Region Asset

Goldman loops over individual fire perimeters and clips the SDD image per fire.
We load the full region boundary here and export one regional raster instead.

In [10]:
bounds_info = region_fc.geometry().bounds().getInfo()
coords      = bounds_info['coordinates'][0]   # outer ring of bbox

lons = [c[0] for c in coords]
lats = [c[1] for c in coords]

xmin, xmax = min(lons), max(lons)
ymin, ymax = min(lats), max(lats)

# Concrete client-side geometry — safe to pass to Export.image.toDrive
region_geom = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])

print(f'Region loaded.')
print(f'  Lon: {xmin:.4f} → {xmax:.4f}')
print(f'  Lat: {ymin:.4f} → {ymax:.4f}')


print('Region loaded.')

Region loaded.
  Lon: -128.9259 → -120.0013
  Lat: 56.0898 → 60.0048
Region loaded.


In [5]:
print(region_fc.size().getInfo())        # should be > 0
print(region_geom.bounds().getInfo())    # should show real coordinates, not 0,0

2
{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-128.92587703972893, 56.089759170078395], [-120.00127162037951, 56.089759170078395], [-120.00127162037951, 60.00478279548971], [-128.92587703972893, 60.00478279548971], [-128.92587703972893, 56.089759170078395]]]}


In [17]:
print(region_geom.bounds().getInfo())    # should show real coordinates, not 0,0

{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-128.92587703972893, 56.08975917007837], [-120.00127162037951, 56.08975917007837], [-120.00127162037951, 60.080037483181655], [-128.92587703972893, 60.080037483181655], [-128.92587703972893, 56.08975917007837]]]}


## Cell 4 — Define `getMetrics(wyr)`

Direct Python port of Goldman's `getMetrics` function.
Every variable name and every operation matches the original JavaScript.

In [29]:
#%%
def getMetrics(wyr):
    """
    Python port of Goldman getMetrics(wyr).
    Water mask and lower_mask removed temporarily to isolate export error.
    """

    start     = str(wyr) + wyStartDay
    end       = str(wyr) + wyEndDay
    start_adv = end
    end_adv   = ee.Date(end).advance(30, 'day')

    adv_days = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start_adv, end_adv)
        .filterBounds(region_geom)
        .size()
        .getInfo()
    )
    print(f'  adv_days = {adv_days}')

    MODIS_cloud_list = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start, end_adv)
        .filterBounds(region_geom)
        .merge(
            ee.ImageCollection('MODIS/061/MYD10A1')
            .filterDate(start, end_adv)
            .filterBounds(region_geom)
        )
        .sort('system:time_start')
        .map(lambda img: img.expression(
            '(BAND == 250)', {'BAND': img.select('NDSI_Snow_Cover_Class')}
        ))
        .toList(400)
    )

    MODIS_snow_list = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start, end_adv)
        .filterBounds(region_geom)
        .merge(
            ee.ImageCollection('MODIS/061/MYD10A1')
            .filterDate(start, end_adv)
            .filterBounds(region_geom)
        )
        .sort('system:time_start')
        .map(lambda img: img.expression(
            '(BAND >= threshold) && (BAND <= 100)',
            {'BAND': img.select('NDSI_Snow_Cover'), 'threshold': NDSI_threshold}
        ))
        .toList(400)
    )

    ndays = MODIS_snow_list.length().getInfo() - adv_days
    print(f'  ndays = {ndays}')

    expected      = 365 + (0 if wyr % 4 else 1)
    sddCorrection = expected - ndays if ndays < expected else 0
    print(f'  sddCorrection = {sddCorrection}')

    sddImage               = ee.Image(0)
    future_snow            = ee.Image(0)
    Snow                   = ee.Image(0)
    accSnow                = ee.Image(0)
    ephemeral_counter      = ee.Image(0)
    noSnowCounter          = ee.Image(0)
    minNoSnow              = ee.Image(minNoSnowdays)
    min_sdays_prior_to_SDD = ee.Image(prior_days)
    sddDetected            = ee.Image(0)

    for current_day in range(ndays + adv_days - 1, -1, -1):

        Cloud       = ee.Image(MODIS_cloud_list.get(current_day)).unmask()
        future_snow = Cloud.And(future_snow.Or(Snow))
        Snow        = ee.Image(MODIS_snow_list.get(current_day)).unmask()

        if current_day < ndays:

            snow_occurrence = future_snow.Or(Snow)
            accSnow         = accSnow.add(snow_occurrence)

            min_sdays_prior_to_SDD = min_sdays_prior_to_SDD.where(
                snow_occurrence.eq(0).And(
                    ephemeral_counter.gt(min_sdays_prior_to_SDD)
                ),
                ephemeral_counter
            )

            ephemeral_counter = (
                ephemeral_counter.add(snow_occurrence).multiply(snow_occurrence)
            )

            noSnowCounter = (
                noSnowCounter
                .add(snow_occurrence.eq(0))
                .multiply(sddDetected.eq(0))
            )

            sddDetected = (
                ephemeral_counter.gt(min_sdays_prior_to_SDD)
                .And(noSnowCounter.gt(minNoSnow))
            )

            minNoSnow = minNoSnow.where(sddDetected, noSnowCounter)

            sdd1     = current_day + sddCorrection
            sdd      = min_sdays_prior_to_SDD.add(ee.Image(sdd1))
            sddImage = sddImage.where(sddDetected, sdd)

    sddImage = sddImage.where(accSnow.eq(ndays - 1), ndays + sddCorrection)

    # ── Water mask ────────────────────────────────────────────────────────────
    # Goldman: GLCF/GLS_WATER — replaced with JRC GlobalSurfaceWater.
    # Do NOT clip the water mask — clipping a differently-projected image
    # before the final SDD clip causes 'Image.clip: Can't transform (N,0.0)'.
    # Apply as a mask only; the clip happens once on the final SDD image.
    water = (
        ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
        .select('seasonality')
        .lt(10)     # 1 = not permanent water, 0 = permanent water
        .unmask(1)  # no-data areas treated as not-water
    )

    # ── SCF image — Goldman: scfImage = accSnow/ndays, masked by water ────────
    # No clip here — mask only
    scfImage = (
        accSnow
        .divide(ndays)
        .updateMask(accSnow)
        .rename('ccSCF')
        .updateMask(water)
    )

    # Goldman: sddImage = sddImage.updateMask(sddImage).rename('SDD').updateMask(water)
    # No clip here — mask only
    sddImage = (
        sddImage
        .updateMask(sddImage)
        .rename('SDD')
        .updateMask(water)
    )

    # Goldman: lower_mask = scfImage.gte(low_value)
    #          SDD = sddImage.updateMask(lower_mask)
    lower_mask = scfImage.gte(ee.Image(low_value))
    SDD        = sddImage.updateMask(lower_mask)

    # Single clip at the very end on the fully masked image
    return (
    SDD
    .reproject(crs='SR-ORG:6974', scale=500)  # MODIS sinusoidal
    .clip(region_geom)
)


print('getMetrics() defined — no water mask, no lower_mask')


getMetrics() defined — no water mask, no lower_mask


## Cell 5 — Run and Export

Goldman loops over fire perimeters and exports one clipped image per fire.
We run `getMetrics()` for each year and export one regional raster to Drive.

In [30]:
#%%
export_tasks = []

for wyr in YEARS:
    print(f'\n{"="*50}')
    print(f'Running getMetrics({wyr})')
    print(f'{"="*50}')

    # Goldman: var snowReturn = getMetrics(wyr)
    snowReturn = getMetrics(wyr)

    # Goldman: Export.image.toDrive({image: snowClip, ...})
    # Image is already clipped inside getMetrics() — pass it directly
    description = f'SDD_{wyr}'
    task = ee.batch.Export.image.toDrive(
        image       = snowReturn,
        description = description,
        folder      = imagesFolder,
        scale       = 500,
        crs         = 'EPSG:4326',
        region      = region_geom,
        maxPixels   = 1e12,
    )
    task.start()
    export_tasks.append((wyr, task))
    print(f'  ✓ Export submitted: {description}  (task {task.id})')

print('\nAll tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')



Running getMetrics(2024)
  adv_days = 30
  ndays = 370
  sddCorrection = 0
  ✓ Export submitted: SDD_2024  (task 3WGKDZDEI5HXTP5UMPFOSIHI)

Running getMetrics(2025)
  adv_days = 30
  ndays = 370
  sddCorrection = 0
  ✓ Export submitted: SDD_2025  (task M47XI5GCCMUY2YILC3KLRY5K)

All tasks submitted.
Monitor at: https://code.earthengine.google.com/tasks


## Cell 6 — Monitor Tasks (optional)

In [26]:
import time

def monitor_tasks(tasks, poll_interval_s=60):
    pending = list(tasks)
    while pending:
        still_running = []
        for wyr, task in pending:
            state = task.status()['state']
            if state == 'COMPLETED':
                print(f'  [{wyr}] ✓ COMPLETED')
            elif state == 'FAILED':
                print(f'  [{wyr}] ✗ FAILED — {task.status().get("error_message", "")}')
            elif state == 'CANCELLED':
                print(f'  [{wyr}] — CANCELLED')
            else:
                print(f'  [{wyr}] {state}')
                still_running.append((wyr, task))
        pending = still_running
        if pending:
            time.sleep(poll_interval_s)
    print('Done.')

monitor_tasks(export_tasks)

  [2024] RUNNING
  [2025] READY
  [2024] ✗ FAILED — Image.clip: Can't transform (7199.0,0.0)
  [2025] ✗ FAILED — Image.clip: Can't transform (4799.0,0.0)
Done.
